# Base vs LoRA Model 对比分析

分析两个模型在 uncommon 图片上的表现差异

In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

In [ ]:
import sys, os
sys.path.append(os.path.join(os.path.abspath(''), '..', '..'))
import config

## 1. 加载数据

In [ ]:
data_dir = config.FOCUS_DATA_DIR

# 加载详细结果
results_base = pd.read_csv(f'{data_dir}/uncommon_all_inference_results_base.csv')
results_lora = pd.read_csv(f'{data_dir}/uncommon_all_inference_results_lora.csv')

# 加载pair统计
stats_base = pd.read_csv(f'{data_dir}/uncommon_pair_statistics_base.csv')
stats_lora = pd.read_csv(f'{data_dir}/uncommon_pair_statistics_lora.csv')

# 加载metrics
with open(f'{data_dir}/uncommon_all_inference_metrics_base.json') as f:
    metrics_base = json.load(f)
with open(f'{data_dir}/uncommon_all_inference_metrics_lora.json') as f:
    metrics_lora = json.load(f)

print(f"Base model results: {len(results_base)} samples")
print(f"LoRA model results: {len(results_lora)} samples")

## 2. 整体性能对比

In [ ]:
# 创建对比表
comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Total Samples', 'Unknown Predictions'],
    'Base Model': [
        f"{metrics_base['accuracy']:.2%}",
        f"{metrics_base['precision']:.4f}",
        f"{metrics_base['recall']:.4f}",
        f"{metrics_base['f1']:.4f}",
        metrics_base['total'],
        metrics_base['unknown']
    ],
    'LoRA Model': [
        f"{metrics_lora['accuracy']:.2%}",
        f"{metrics_lora['precision']:.4f}",
        f"{metrics_lora['recall']:.4f}",
        f"{metrics_lora['f1']:.4f}",
        metrics_lora['total'],
        metrics_lora['unknown']
    ]
})

print("="*60)
print("Overall Performance Comparison")
print("="*60)
display(comparison)

In [ ]:
# 可视化整体性能
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
base_values = [metrics_base['accuracy'], metrics_base['precision'], metrics_base['recall'], metrics_base['f1']]
lora_values = [metrics_lora['accuracy'], metrics_lora['precision'], metrics_lora['recall'], metrics_lora['f1']]

x = np.arange(len(metrics_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, base_values, width, label='Base Model', color='steelblue')
bars2 = ax.bar(x + width/2, lora_values, width, label='LoRA Model', color='coral')

ax.set_ylabel('Score')
ax.set_title('Overall Performance Comparison: Base vs LoRA')
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.legend()
ax.set_ylim(0, 1.1)

# 添加数值标签
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.2%}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=10)
for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.2%}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=10)

plt.tight_layout()
# plt.savefig(f'{data_dir}/overall_comparison.png', dpi=150)
plt.show()

## 3. Category-Location Pair 对比分析

In [ ]:
# 合并两个模型的统计
stats_base['model'] = 'Base'
stats_lora['model'] = 'LoRA'

# 创建pair标识
stats_base['pair'] = stats_base['category'] + '-' + stats_base['location']
stats_lora['pair'] = stats_lora['category'] + '-' + stats_lora['location']

# Merge for comparison
pair_comparison = stats_base[['pair', 'category', 'location', 'total', 'out_of_context_rate', 'accuracy']].merge(
    stats_lora[['pair', 'out_of_context_rate', 'accuracy']],
    on='pair',
    suffixes=('_base', '_lora')
)

# 计算差异
pair_comparison['ooc_rate_diff'] = pair_comparison['out_of_context_rate_lora'] - pair_comparison['out_of_context_rate_base']
pair_comparison['accuracy_diff'] = pair_comparison['accuracy_lora'] - pair_comparison['accuracy_base']

print(f"Total pairs: {len(pair_comparison)}")
pair_comparison.head(10)

In [ ]:
# 完整的pair对比表
display_df = pair_comparison[['pair', 'total', 'out_of_context_rate_base', 'out_of_context_rate_lora', 
                               'ooc_rate_diff', 'accuracy_base', 'accuracy_lora', 'accuracy_diff']].copy()
display_df.columns = ['Pair', 'Total', 'OOC Rate (Base)', 'OOC Rate (LoRA)', 'OOC Diff', 
                      'Acc (Base)', 'Acc (LoRA)', 'Acc Diff']

# 按OOC Rate差异排序
display_df = display_df.sort_values('OOC Diff', ascending=False)

print("="*80)
print("Category-Location Pair Comparison (sorted by OOC Rate difference)")
print("="*80)

# 格式化显示
for col in ['OOC Rate (Base)', 'OOC Rate (LoRA)', 'OOC Diff', 'Acc (Base)', 'Acc (LoRA)', 'Acc Diff']:
    display_df[col] = display_df[col].apply(lambda x: f"{x:.2%}")

display(display_df)

## 4. LoRA模型改进最大/最小的Pairs

In [ ]:
# 重新获取数值版本
pair_comparison_sorted = pair_comparison.sort_values('ooc_rate_diff', ascending=False)

print("="*60)
print("🔺 Top 5: LoRA 更倾向于判断为 Out-of-context (vs Base)")
print("="*60)
for i, (_, row) in enumerate(pair_comparison_sorted.head(5).iterrows()):
    print(f"{i+1}. {row['pair']:20s} | Base: {row['out_of_context_rate_base']:.2%} → LoRA: {row['out_of_context_rate_lora']:.2%} (Δ {row['ooc_rate_diff']:+.2%})")

print("\n" + "="*60)
print("🔻 Top 5: LoRA 更倾向于判断为 In-context (vs Base)")
print("="*60)
for i, (_, row) in enumerate(pair_comparison_sorted.tail(5).iloc[::-1].iterrows()):
    print(f"{i+1}. {row['pair']:20s} | Base: {row['out_of_context_rate_base']:.2%} → LoRA: {row['out_of_context_rate_lora']:.2%} (Δ {row['ooc_rate_diff']:+.2%})")

In [ ]:
# 按accuracy差异排序
pair_comparison_acc = pair_comparison.sort_values('accuracy_diff', ascending=False)

print("="*60)
print("✅ Top 5: LoRA 准确率提升最大的 Pairs")
print("="*60)
for i, (_, row) in enumerate(pair_comparison_acc.head(5).iterrows()):
    print(f"{i+1}. {row['pair']:20s} | Base: {row['accuracy_base']:.2%} → LoRA: {row['accuracy_lora']:.2%} (Δ {row['accuracy_diff']:+.2%})")

print("\n" + "="*60)
print("❌ Top 5: LoRA 准确率下降最大的 Pairs")
print("="*60)
for i, (_, row) in enumerate(pair_comparison_acc.tail(5).iloc[::-1].iterrows()):
    print(f"{i+1}. {row['pair']:20s} | Base: {row['accuracy_base']:.2%} → LoRA: {row['accuracy_lora']:.2%} (Δ {row['accuracy_diff']:+.2%})")

## 5. 热力图可视化

In [ ]:
# 创建热力图数据
categories = pair_comparison['category'].unique()
locations = pair_comparison['location'].unique()

# Base model OOC rate heatmap
heatmap_base = pair_comparison.pivot(index='category', columns='location', values='out_of_context_rate_base')
heatmap_lora = pair_comparison.pivot(index='category', columns='location', values='out_of_context_rate_lora')
heatmap_diff = pair_comparison.pivot(index='category', columns='location', values='ooc_rate_diff')

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Base model
sns.heatmap(heatmap_base, annot=True, fmt='.0%', cmap='RdYlGn', ax=axes[0], 
            vmin=0, vmax=1, cbar_kws={'label': 'OOC Rate'})
axes[0].set_title('Base Model: Out-of-Context Rate', fontsize=14)
axes[0].set_xlabel('Location')
axes[0].set_ylabel('Category')

# LoRA model
sns.heatmap(heatmap_lora, annot=True, fmt='.0%', cmap='RdYlGn', ax=axes[1],
            vmin=0, vmax=1, cbar_kws={'label': 'OOC Rate'})
axes[1].set_title('LoRA Model: Out-of-Context Rate', fontsize=14)
axes[1].set_xlabel('Location')
axes[1].set_ylabel('Category')

# Difference (LoRA - Base)
sns.heatmap(heatmap_diff, annot=True, fmt='+.0%', cmap='RdBu_r', ax=axes[2],
            center=0, cbar_kws={'label': 'Difference'})
axes[2].set_title('Difference (LoRA - Base)', fontsize=14)
axes[2].set_xlabel('Location')
axes[2].set_ylabel('Category')

plt.tight_layout()
# plt.savefig(f'{data_dir}/heatmap_comparison.png', dpi=150)
plt.show()

## 6. 按 Category 汇总分析

In [ ]:
# 按category汇总
cat_summary_base = results_base.groupby('category').agg(
    total=('correct', 'count'),
    correct=('correct', 'sum'),
    ooc_pred=('prediction', lambda x: (x == 'Out-of-context').sum())
).reset_index()
cat_summary_base['accuracy'] = cat_summary_base['correct'] / cat_summary_base['total']
cat_summary_base['ooc_rate'] = cat_summary_base['ooc_pred'] / cat_summary_base['total']

cat_summary_lora = results_lora.groupby('category').agg(
    total=('correct', 'count'),
    correct=('correct', 'sum'),
    ooc_pred=('prediction', lambda x: (x == 'Out-of-context').sum())
).reset_index()
cat_summary_lora['accuracy'] = cat_summary_lora['correct'] / cat_summary_lora['total']
cat_summary_lora['ooc_rate'] = cat_summary_lora['ooc_pred'] / cat_summary_lora['total']

# 合并
cat_comparison = cat_summary_base[['category', 'total', 'accuracy', 'ooc_rate']].merge(
    cat_summary_lora[['category', 'accuracy', 'ooc_rate']],
    on='category',
    suffixes=('_base', '_lora')
)
cat_comparison['acc_diff'] = cat_comparison['accuracy_lora'] - cat_comparison['accuracy_base']
cat_comparison['ooc_diff'] = cat_comparison['ooc_rate_lora'] - cat_comparison['ooc_rate_base']

print("Per-Category Summary:")
display(cat_comparison.sort_values('acc_diff', ascending=False))

In [ ]:
# 按category的准确率对比图
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(cat_comparison))
width = 0.35

cat_comparison_sorted = cat_comparison.sort_values('accuracy_base', ascending=False)

bars1 = ax.bar(x - width/2, cat_comparison_sorted['accuracy_base'], width, label='Base Model', color='steelblue')
bars2 = ax.bar(x + width/2, cat_comparison_sorted['accuracy_lora'], width, label='LoRA Model', color='coral')

ax.set_ylabel('Accuracy')
ax.set_title('Accuracy by Category: Base vs LoRA')
ax.set_xticks(x)
ax.set_xticklabels(cat_comparison_sorted['category'], rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1.1)

plt.tight_layout()
# plt.savefig(f'{data_dir}/category_accuracy_comparison.png', dpi=150)
plt.show()

## 7. 按 Location 汇总分析

In [ ]:
# 按location汇总
loc_summary_base = results_base.groupby('location').agg(
    total=('correct', 'count'),
    correct=('correct', 'sum'),
    ooc_pred=('prediction', lambda x: (x == 'Out-of-context').sum())
).reset_index()
loc_summary_base['accuracy'] = loc_summary_base['correct'] / loc_summary_base['total']
loc_summary_base['ooc_rate'] = loc_summary_base['ooc_pred'] / loc_summary_base['total']

loc_summary_lora = results_lora.groupby('location').agg(
    total=('correct', 'count'),
    correct=('correct', 'sum'),
    ooc_pred=('prediction', lambda x: (x == 'Out-of-context').sum())
).reset_index()
loc_summary_lora['accuracy'] = loc_summary_lora['correct'] / loc_summary_lora['total']
loc_summary_lora['ooc_rate'] = loc_summary_lora['ooc_pred'] / loc_summary_lora['total']

# 合并
loc_comparison = loc_summary_base[['location', 'total', 'accuracy', 'ooc_rate']].merge(
    loc_summary_lora[['location', 'accuracy', 'ooc_rate']],
    on='location',
    suffixes=('_base', '_lora')
)
loc_comparison['acc_diff'] = loc_comparison['accuracy_lora'] - loc_comparison['accuracy_base']
loc_comparison['ooc_diff'] = loc_comparison['ooc_rate_lora'] - loc_comparison['ooc_rate_base']

print("Per-Location Summary:")
display(loc_comparison.sort_values('acc_diff', ascending=False))

In [ ]:
# 按location的准确率对比图
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(loc_comparison))
width = 0.35

loc_comparison_sorted = loc_comparison.sort_values('accuracy_base', ascending=False)

bars1 = ax.bar(x - width/2, loc_comparison_sorted['accuracy_base'], width, label='Base Model', color='steelblue')
bars2 = ax.bar(x + width/2, loc_comparison_sorted['accuracy_lora'], width, label='LoRA Model', color='coral')

ax.set_ylabel('Accuracy')
ax.set_title('Accuracy by Location: Base vs LoRA')
ax.set_xticks(x)
ax.set_xticklabels(loc_comparison_sorted['location'], rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1.1)

plt.tight_layout()
# plt.savefig(f'{data_dir}/location_accuracy_comparison.png', dpi=150)
plt.show()

## 8. 预测一致性分析

In [ ]:
# 合并两个模型的预测结果
merged = results_base[['filename', 'category', 'location', 'ground_truth', 'prediction', 'correct']].merge(
    results_lora[['filename', 'prediction', 'correct']],
    on='filename',
    suffixes=('_base', '_lora')
)

# 计算一致性
merged['both_correct'] = merged['correct_base'] & merged['correct_lora']
merged['both_wrong'] = ~merged['correct_base'] & ~merged['correct_lora']
merged['base_only_correct'] = merged['correct_base'] & ~merged['correct_lora']
merged['lora_only_correct'] = ~merged['correct_base'] & merged['correct_lora']
merged['same_prediction'] = merged['prediction_base'] == merged['prediction_lora']

print("="*60)
print("Prediction Consistency Analysis")
print("="*60)
print(f"Total samples: {len(merged)}")
print(f"Both correct: {merged['both_correct'].sum()} ({merged['both_correct'].mean():.2%})")
print(f"Both wrong: {merged['both_wrong'].sum()} ({merged['both_wrong'].mean():.2%})")
print(f"Base only correct: {merged['base_only_correct'].sum()} ({merged['base_only_correct'].mean():.2%})")
print(f"LoRA only correct: {merged['lora_only_correct'].sum()} ({merged['lora_only_correct'].mean():.2%})")
print(f"Same prediction: {merged['same_prediction'].sum()} ({merged['same_prediction'].mean():.2%})")

In [ ]:
# 一致性可视化
consistency_data = [
    merged['both_correct'].sum(),
    merged['both_wrong'].sum(),
    merged['base_only_correct'].sum(),
    merged['lora_only_correct'].sum()
]
consistency_labels = ['Both Correct', 'Both Wrong', 'Base Only Correct', 'LoRA Only Correct']
colors = ['green', 'red', 'steelblue', 'coral']

fig, ax = plt.subplots(figsize=(8, 8))
wedges, texts, autotexts = ax.pie(consistency_data, labels=consistency_labels, autopct='%1.1f%%',
                                   colors=colors, explode=(0.02, 0.02, 0.02, 0.02))
ax.set_title('Prediction Consistency: Base vs LoRA', fontsize=14)

plt.tight_layout()
# plt.savefig(f'{data_dir}/prediction_consistency.png', dpi=150)
plt.show()

## 9. 错误案例分析

In [ ]:
# LoRA改进的案例（Base错，LoRA对）
lora_improved = merged[merged['lora_only_correct']]
print(f"LoRA improved cases: {len(lora_improved)}")
print("\nDistribution by category-location:")
print(lora_improved.groupby(['category', 'location']).size().sort_values(ascending=False).head(10))

In [ ]:
# LoRA退化的案例（Base对，LoRA错）
lora_degraded = merged[merged['base_only_correct']]
print(f"LoRA degraded cases: {len(lora_degraded)}")
print("\nDistribution by category-location:")
print(lora_degraded.groupby(['category', 'location']).size().sort_values(ascending=False).head(10))

In [ ]:
# 两个模型都错的案例
both_wrong_cases = merged[merged['both_wrong']]
print(f"Both models wrong: {len(both_wrong_cases)}")
print("\nDistribution by category-location:")
print(both_wrong_cases.groupby(['category', 'location']).size().sort_values(ascending=False).head(10))

## 10. 保存分析结果

In [ ]:
# # 保存pair对比结果
# pair_comparison.to_csv(f'{data_dir}/pair_comparison_base_vs_lora.csv', index=False)

# # 保存category对比结果
# cat_comparison.to_csv(f'{data_dir}/category_comparison_base_vs_lora.csv', index=False)

# # 保存location对比结果
# loc_comparison.to_csv(f'{data_dir}/location_comparison_base_vs_lora.csv', index=False)

# # 保存一致性分析结果
# merged.to_csv(f'{data_dir}/prediction_consistency_analysis.csv', index=False)

# print("✅ Analysis results saved:")
# print(f"   - {data_dir}/pair_comparison_base_vs_lora.csv")
# print(f"   - {data_dir}/category_comparison_base_vs_lora.csv")
# print(f"   - {data_dir}/location_comparison_base_vs_lora.csv")
# print(f"   - {data_dir}/prediction_consistency_analysis.csv")
# print(f"\n📊 Figures saved:")
# print(f"   - {data_dir}/overall_comparison.png")
# print(f"   - {data_dir}/heatmap_comparison.png")
# print(f"   - {data_dir}/category_accuracy_comparison.png")
# print(f"   - {data_dir}/location_accuracy_comparison.png")
# print(f"   - {data_dir}/prediction_consistency.png")

## 11. 总结

In [ ]:
print("="*80)
print("SUMMARY")
print("="*80)

print(f"\n📊 Overall Performance:")
print(f"   Base Model Accuracy: {metrics_base['accuracy']:.2%}")
print(f"   LoRA Model Accuracy: {metrics_lora['accuracy']:.2%}")
print(f"   Improvement: {metrics_lora['accuracy'] - metrics_base['accuracy']:+.2%}")

print(f"\n📈 Key Findings:")
print(f"   - Total pairs analyzed: {len(pair_comparison)}")
print(f"   - Pairs where LoRA improved: {(pair_comparison['accuracy_diff'] > 0).sum()}")
print(f"   - Pairs where LoRA degraded: {(pair_comparison['accuracy_diff'] < 0).sum()}")
print(f"   - Pairs with no change: {(pair_comparison['accuracy_diff'] == 0).sum()}")

print(f"\n🔍 Prediction Consistency:")
print(f"   - Same prediction rate: {merged['same_prediction'].mean():.2%}")
print(f"   - LoRA improved: {merged['lora_only_correct'].sum()} samples")
print(f"   - LoRA degraded: {merged['base_only_correct'].sum()} samples")
print(f"   - Net improvement: {merged['lora_only_correct'].sum() - merged['base_only_correct'].sum()} samples")